# Spinning plates 🧪

Dr Pippa Petts has emailed in a panic: her compound screen on 96-well plate
EXP-042 came back from the plate reader as an export that won't even open in
Excel, the layout is in a separate file, and the controls are not behaving. Can
you find out whether the experiment worked? Read her email and follow the
[problem](../problems/data-analysis/spinning-plates.md).

Implement the helper functions in
[plate_reader.py](../src/pyproblems/plate_reader.py) as you reach each stage and
check them with `uv run pytest tests/test_plate_reader.py -q`; use this notebook
to call them, make plots and record what you find.

Run the setup cell first. It sets `data_dir`, the folder
`src/pyproblems/datasets/` that holds the three files for this plate:

- `plate_reader_export.txt`: the plate reader's export
- `plate_layout.csv`: the layout of the plate as designed
- `plate_run_record.json`: the run record kept alongside the run

Reading the export and the layout is your first job.

In [ ]:
import json
from importlib.resources import files

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pyproblems.plate_reader import (
    join_layout,
    mark_usable,
    normalise_to_reference,
    parse_plate_reader,
    percent_of_vehicle,
    pivot_channels,
    plate_heatmap,
    plot_concentration_response,
    remap_readings,
    subtract_blank,
    summarise_replicates,
    validate_plate,
    z_prime,
)

data_dir = files("pyproblems.datasets")

## The experiment

Three compounds (A, B and C) were each tested at four concentrations, with four
replicate wells per concentration, alongside three kinds of control:

| `sample_type` | What is in the well | Biomarker | Reference |
| --- | --- | --- | --- |
| `blank` | medium, no cells | background only | background only |
| `vehicle` | cells and DMSO only (uninhibited) | high | high |
| `inhibitor` | cells and a reference inhibitor | strongly reduced | high |


Each well was read in two channels. The **biomarker** is the signal of interest.
The **reference** is proportional to the amount of cellular material in the
well, however many cells happened to be seeded there.

Before you open the data, write down a prediction: if the experiment worked,
what should each control look like in each channel, and what would a convincing
concentration-response look like?

My prediction:



## 1. Excel failed, python to the rescue 🦸

Read the export as the instrument wrote it. What
would a parser have to cope with? Then implement `parse_plate_reader` and
`validate_plate`
(`uv run pytest tests/test_plate_reader.py -k "parse or validate" -q`) and call
them below.

In [ ]:
# TODO: read the export and the layout in data_dir.
# The export is text and the layout is a CSV table.
export_text = None
layout = None

In [ ]:
# Look at the export exactly as the instrument wrote it.
print(export_text[:2500])

In [ ]:
# TODO: parse the export and validate it. Which readings carry a QC flag?
readings = parse_plate_reader(export_text)
validate_plate(readings)
readings["qc_flag"].value_counts()

## 2. One well, one row

[Tidy data](https://byuidatascience.github.io/python4ds/tidy-data.html) has one
observation per row and one variable per column. The export might be readable by
eye, but for analysis you want one row per *well*.

Implement `pivot_channels` and `join_layout`
(`uv run pytest tests/test_plate_reader.py -k "pivot or join" -q`), then build
one analysis-ready table: a row per well, with both channels, their QC flags and
the layout. Keep the raw readings unchanged; derived values go in new columns.

In [ ]:
# TODO: one row per well, then join the layout.
wells = pivot_channels(readings)
table = join_layout(wells, layout)
table.head()

## 3. Out of control

Compare what you measured with what you expected from the briefing.

A plate heatmap shows measurements in the same layout as they would be on a
plate, which makes spatial problems far easier to see. The pictures below show
the *intended layout* beside the two channels, with row A at the top. Then
summarise the controls by group.

The control measurements may look reasonable one at a time, yet fail to match
the wells they were supposedly in. What could explain this?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
plate_heatmap(table, "sample_type", axes[0], title="Intended layout")
plate_heatmap(table, "biomarker", axes[1], title="Biomarker", cmap="viridis")
plate_heatmap(table, "reference", axes[2], title="Reference", cmap="viridis")
plt.tight_layout()

In [ ]:
# TODO: summarise the control wells by sample_type. Do the signals look like their labels?
controls = table[table["sample_type"] != "treatment"]
controls.groupby("sample_type")[["biomarker", "reference"]].agg(
    ["mean", "std", "count"]
)

Describe what you see. Which controls behave as labelled, and which do not?



### Competing explanations

Write down any explanations you can think of for what you see, and how you
might tell them apart.

One tool you can use: `remap_readings` reassigns readings to different wells
(`uv run pytest tests/test_plate_reader.py -k remap -q`). A mapping is a
function that takes a layout well such as `"C03"` and returns the well whose
reading you think belongs to it.

Decide between your explanations *before* you look at what each one does to the
compounds' dose-response curves: never choose by which one makes a compound
look best.

In [ ]:
# TODO: add your own candidate mappings.
hypotheses = {
    "as exported": lambda well: well,
}

fig, axes = plt.subplots(
    1, len(hypotheses) + 1, figsize=(5 * (len(hypotheses) + 1), 4), squeeze=False
)
plate_heatmap(table, "sample_type", axes[0, 0], title="Intended layout")
for ax, (name, mapping) in zip(axes[0, 1:], hypotheses.items(), strict=True):
    remapped = remap_readings(table, mapping)
    plate_heatmap(remapped, "reference", ax, title=name, cmap="viridis")
plt.tight_layout()

In [ ]:
# TODO: gather and show your evidence for your plausible hypotheses.

Which hypothesis do you choose, and what is the evidence? How decisive is it?



Now that you have a hypothesis, is there independent evidence of what happened?
Load the run record. The lab's standard procedure is to load plates with well
A1 at the front-left of the tray. Compare that with the run record.

In [ ]:
run_record = json.loads(
    (data_dir / "plate_run_record.json").read_text(encoding="utf-8")
)
run_record

What does the run record say, and how well does it fit?



## 4. The Pipeline

Process the readings with the mapping you chose by chaining functions together
in a **pipeline**: each step takes a DataFrame and returns a modified
DataFrame, which becomes the input to the next step. pandas chains them with
`DataFrame.pipe`.

Implement `mark_usable`, `subtract_blank`, `normalise_to_reference`,
`percent_of_vehicle` and `summarise_replicates` (`uv run pytest
tests/test_plate_reader.py -k "usable or blank or normalise or vehicle or
summarise or pipeline" -q`). Together they turn the corrected readings into a
percentage-of-vehicle response for each compound and concentration:

1. decide which wells you will not use, and why (your QC policy);
2. subtract the blank level in each channel;
3. divide the biomarker by the reference, so cell-loading differences cancel;
4. express wells relative to the vehicle controls;
5. summarise the replicates.

The first three are chained steps. `percent_of_vehicle` returns a column to
attach with `assign`, and `summarise_replicates` ends the pipeline with a
summary table.

Then check the controls again after correction: do they meet reasonable QC
expectations? `z_prime` is not a pipeline step: it takes two groups of values
(for example the vehicle and inhibitor wells) and returns one number, the
[Z′-factor](https://en.wikipedia.org/wiki/Z-factor), which measures how well
they separate. Above about 0.5 is excellent; near zero or negative is poor.
Implement it too (`uv run pytest tests/test_plate_reader.py -k z_prime -q`).

In [ ]:
# TODO: set chosen to the mapping you have evidence for.
chosen = None

processed = (
    table.pipe(remap_readings, chosen)
    .pipe(mark_usable)
    .pipe(subtract_blank)
    .pipe(normalise_to_reference)
)
processed.head()

In [ ]:
# TODO: which wells were excluded, and why? How well do the corrected controls separate?
processed.loc[
    ~processed["usable"], ["well", "sample_type", "biomarker_flag", "reference_flag"]
]

Now compare four analyses of the same readings, drawn as a 2 × 2 grid of
concentration-response plots (mean ± SEM, as a percentage of the vehicle).

The **rows** are the two mappings:

- *as exported*: the readings exactly as the instrument reported them;
- *my choice*: the readings remapped with `chosen`.

The **columns** are the two signals analysed:

- *blank-subtracted*: the biomarker with the background removed
  (`biomarker_net`);
- *normalised*: the blank-subtracted biomarker divided by the blank-subtracted
  reference (`normalised`).

Interpret the differences rather than just looking at the plots:

- **Rows:** what changes when you correct the orientation, and what is the
  as-exported analysis actually showing?
- **Columns:** what changes when you normalise to the reference? Compare the
  replicate variability (for example the SEM of each condition) with and without
  normalising. By how much does it change, and why?
- For each analysis you would not show a colleague, explain how it could mislead
  them.

Which of the four would you want to show a colleague? Which corrections did you
need? Did any of the compounds do anything?

In [ ]:
def response_summary(mapping, column):
    wells = (
        table.pipe(remap_readings, mapping)
        .pipe(mark_usable)
        .pipe(subtract_blank)
        .pipe(normalise_to_reference)
    )
    percent = percent_of_vehicle(wells, column)
    return summarise_replicates(wells.assign(percent=percent), "percent")


fig, axes = plt.subplots(2, 2, figsize=(11, 8), sharex=True)
for i, (orientation, mapping) in enumerate(
    [("as exported", hypotheses["as exported"]), ("my choice", chosen)]
):
    for j, (signal, column) in enumerate(
        [("blank-subtracted", "biomarker_net"), ("normalised", "normalised")]
    ):
        summary = response_summary(mapping, column)
        plot_concentration_response(summary, axes[i, j], f"{orientation}, {signal}")
        axes[i, j].set_ylabel("% of vehicle")
plt.tight_layout()

In [ ]:
# TODO: compare the replicate variability (sem) of each compound with and without
# normalisation, using response_summary with your chosen mapping.

Describe the response of each compound. Is it consistent across replicate wells?
How did the replicate variability change when you normalised, and why? Which of
the other analyses are misleading, and how?



## 5. Would you sign off on this?

Your colleague wants to progress Compound A to the next stage. Would you
recommend **progressing** it, **repeating** the experiment or **rejecting** the
result?

Write a short conclusion, supported by your plots and QC findings. Consider:

- How convincingly can you establish what if anything went wrong?
- What evidence is there for what may have happened?
- Do the controls meet reasonable QC expectations?
- Are responses consistent across replicate wells?
- What uncertainty remains about sample identities and instrument handling?
- What confirmatory experiment would you recommend?

My conclusion:

